# Signal Export Guide

Assemble a clean, time-indexed signal DataFrame from all pipeline outputs
and export it for downstream use (backtesting, spreadsheets, R).

**No API key required.**

**Output columns:**
- `sentiment_{sector}` — daily mean score (−1/0/+1) per sector (19 cols)
- `entity_mentions_{entity}` — daily mention count for top entities
- `topic_spike` — 1 if any topic spiked that day, 0 otherwise
- `n_spikes` — number of spikes that day

## Sections
1. [Configuration](#1-configuration)
2. [Setup](#2-setup)
3. [Sector signals](#3-sector-signals)
4. [Entity signals](#4-entity-signals)
5. [Topic spike flags](#5-topic-spike-flags)
6. [Assemble & export](#6-assemble--export)

---
## 1 — Configuration

In [ ]:
LOOKBACK       = 90        # rolling window in days
TOP_ENTITIES   = 20        # how many top entities to include as columns
OUTPUT_TSV     = "../data/signals.tsv"      # where to write the signal file
OUTPUT_PARQUET = "../data/signals.parquet"  # optional parquet export

---
## 2 — Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..") if Path("../pipeline").exists() else Path(".")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.query_sector import get_all_sectors_pivot
from pipeline.query_entity import get_all_entities_ts
from pipeline.constants import TOPIC_TRENDS_FILE, SECTOR_SUMMARY_FILE

# Pull sector_summary.tsv from HuggingFace if missing.
if not SECTOR_SUMMARY_FILE.exists():
    import os
    from dotenv import load_dotenv
    from datasets import load_dataset
    load_dotenv(PROJECT_ROOT / ".env")
    hf_user = os.environ.get("HUGGINGFACE_REPO", "lacetohf/feeds").split("/")[0]
    ds = load_dataset(f"{hf_user}/sector-analysis", split="train")
    SECTOR_SUMMARY_FILE.parent.mkdir(parents=True, exist_ok=True)
    ds.to_pandas().to_csv(SECTOR_SUMMARY_FILE, sep="\t", index=False)
    print(f"Pulled sector-analysis from HuggingFace -> {SECTOR_SUMMARY_FILE}")

print(f"sector_summary.tsv : {SECTOR_SUMMARY_FILE.exists()}")
print(f"topic_trends.tsv   : {TOPIC_TRENDS_FILE.exists()}")

---
## 3 — Sector signals

Wide pivot: one row per date, one column per sector, value = mean daily
sentiment score. NaN = sector not seen that day.

In [ ]:
pivot = get_all_sectors_pivot(lookback_days=LOOKBACK, freq="D")
# Prefix columns for clarity in the final signal file.
pivot.columns = [f"sentiment_{c}" for c in pivot.columns]

print(f"Shape      : {pivot.shape}  (dates x sectors)")
print(f"Date range : {pivot.index.min().date()} -> {pivot.index.max().date()}")
print(f"NaN %      : {pivot.isna().mean().mean():.1%}")
pivot.tail(3)

---
## 4 — Entity signals

Daily mention counts for the top-N most-mentioned entities.
One column per entity: `entity_mentions_{name}`.

In [ ]:
df_entities = get_all_entities_ts(lookback_days=LOOKBACK)

# Pick top entities by total mentions.
top_ent_names = (
    df_entities.groupby("entity")["sentiment_score"]
    .count()
    .nlargest(TOP_ENTITIES)
    .index.tolist()
)

# Daily mention count per entity (count of rows = count of sector mentions).
entity_daily = (
    df_entities[df_entities["entity"].isin(top_ent_names)]
    .groupby(["date", "entity"])
    .size()
    .unstack("entity")
    .fillna(0)
    .astype(int)
)
entity_daily.columns = [f"entity_mentions_{c}" for c in entity_daily.columns]

print(f"Entity signal shape : {entity_daily.shape}")
entity_daily.tail(3)

---
## 5 — Topic spike flags

Binary flag: did any topic spike on this date? Also include raw spike count.

In [ ]:
from datetime import datetime as _dt, timedelta

topic_flags = []

if TOPIC_TRENDS_FILE.exists():
    trends_df = pd.read_csv(TOPIC_TRENDS_FILE, sep="\t")
    from pipeline.cluster_topics import get_emerging_topics

    cutoff_date = (pd.Timestamp.today() - pd.Timedelta(days=LOOKBACK)).date()
    for d in sorted(trends_df["date"].unique()):
        if _dt.strptime(d, "%Y-%m-%d").date() < cutoff_date:
            continue
        dt = _dt.strptime(d, "%Y-%m-%d").date()
        n  = len(get_emerging_topics(dt, trends_df))
        topic_flags.append({"date": pd.Timestamp(d), "n_spikes": n, "topic_spike": int(n > 0)})

    df_spikes = pd.DataFrame(topic_flags).set_index("date")
    print(f"Topic flag shape : {df_spikes.shape}")
    print(f"Spike days       : {df_spikes['topic_spike'].sum()} / {len(df_spikes)}")
    df_spikes.tail(3)
else:
    print("topic_trends.tsv missing — topic spike flags will be omitted from the export.")
    df_spikes = pd.DataFrame()

---
## 6 — Assemble & export

Join all three signal sources on date index, forward-fill sector NaNs (carry
last known score through gaps), then export to TSV and optionally Parquet.

In [ ]:
from pathlib import Path

# Join on date index.
frames = [pivot]
if not entity_daily.empty:
    frames.append(entity_daily)
if not df_spikes.empty:
    frames.append(df_spikes)

signals = (
    pd.concat(frames, axis=1)
    .sort_index()
)

# Forward-fill sector scores through weekends / gaps (carry last known score).
sector_cols = [c for c in signals.columns if c.startswith("sentiment_")]
signals[sector_cols] = signals[sector_cols].ffill()

# Fill entity mention NaNs with 0 (entity not seen = 0 mentions).
entity_cols = [c for c in signals.columns if c.startswith("entity_mentions_")]
signals[entity_cols] = signals[entity_cols].fillna(0).astype(int)

# Fill spike flags with 0 for dates outside topic_trends.tsv range.
if "topic_spike" in signals.columns:
    signals[["topic_spike", "n_spikes"]] = signals[["topic_spike", "n_spikes"]].fillna(0).astype(int)

print(f"Signal DataFrame shape : {signals.shape}")
print(f"Columns                : {len(sector_cols)} sector  +  {len(entity_cols)} entity  +  spike flags")
print(f"Date range             : {signals.index.min().date()} -> {signals.index.max().date()}")
print(f"Remaining NaN %        : {signals.isna().mean().mean():.2%}")
signals.tail(3)

In [ ]:
# Export to TSV (always) and Parquet (if pyarrow/fastparquet is installed).
out_tsv = Path(OUTPUT_TSV)
out_tsv.parent.mkdir(parents=True, exist_ok=True)
signals.to_csv(out_tsv, sep="\t")
print(f"Saved TSV     : {out_tsv}  ({out_tsv.stat().st_size // 1024} KB)")

try:
    out_pq = Path(OUTPUT_PARQUET)
    signals.to_parquet(out_pq)
    print(f"Saved Parquet : {out_pq}  ({out_pq.stat().st_size // 1024} KB)")
except ImportError:
    print("Parquet skipped — install pyarrow:  pip install pyarrow")

print(f"\nTo load in Python:")
print(f"  df = pd.read_csv('{out_tsv}', sep='\\t', index_col=0, parse_dates=True)")
print(f"\nTo load in R:")
print(f"  df <- read.delim('{out_tsv}', sep='\\t')")